# Lending Club Data Modeling

This notebook loads cleaned datasets and preprocessing artifacts from the data wrangling stage.

It provides:
- Training of multiple classification models (Logistic Regression and XGBoost)
- Comparison between fundamental and full feature sets
- Proper encoding and preprocessing pipelines for modeling
- Time-based validation and test evaluation
- Performance evaluation using ROC-AUC, PR-AUC, Brier score, and F1-score
- Threshold optimization based on validation data
- Export of trained models, encoders, predictions, and evaluation summaries for visualization

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from xgboost import XGBClassifier

In [2]:
# Load artifacts from wrangling step
ARTIFACT_DIR = Path("artifacts/cleaning_only")
MODEL_DIR = Path("artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_OUTPUTS_PATH = ARTIFACT_DIR / "cleaned_outputs.joblib"
META_PATH = ARTIFACT_DIR / "meta.joblib"

if not CLEANED_OUTPUTS_PATH.exists():
    raise FileNotFoundError(f"Missing: {CLEANED_OUTPUTS_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"Missing: {META_PATH}")

meta = joblib.load(META_PATH)
cleaned_outputs = joblib.load(CLEANED_OUTPUTS_PATH)

print("Loaded wrangling artifacts.")
print("Meta:", meta)
print("Target note:", meta.get("target_definition_note", "N/A"))

Loaded wrangling artifacts.
Meta: {'snapshot_date': '2018-12-31', 'horizon_months': 12, 'split_years': {'train_end_year': 2015, 'val_year': 2016, 'test_year': 2017}, 'fundamental_drop': ['grade', 'sub_grade', 'int_rate', 'installment'], 'non_default_final_statuses': ['Does not meet the credit policy. Status: Fully Paid', 'Fully Paid'], 'active_nondefault_statuses': ['Current', 'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)'], 'default_statuses': ['Charged Off', 'Default', 'Does not meet the credit policy. Status: Charged Off'], 'include_addr_state': True, 'include_vintage_year': True, 'target_definition_note': 'default_12m_proxy: target=1 if loan has default status and last_pymnt_d <= issue_d + horizon; target=0 otherwise among eligible loans. Eligible means horizon_end <= snapshot_date. Default rows with missing last_pymnt_d are dropped as uncertain.'}
Target note: default_12m_proxy: target=1 if loan has default status and last_pymnt_d <= issue_d + horizon; target=0 other

In [3]:
#  Unpack cleaned datasets + labels
# Fundamental (logistic-clean version from wrangling)
X_train_fund_clean = cleaned_outputs["X_train_fund_clean"]
X_val_fund_clean = cleaned_outputs["X_val_fund_clean"]
X_test_fund_clean = cleaned_outputs["X_test_fund_clean"]

# Fundamental (xgb version from wrangling)
X_train_fund_xgb = cleaned_outputs["X_train_fund_xgb"]
X_val_fund_xgb = cleaned_outputs["X_val_fund_xgb"]
X_test_fund_xgb = cleaned_outputs["X_test_fund_xgb"]

# Full xgb
X_train_full_xgb = cleaned_outputs["X_train_full_xgb"]
X_val_full_xgb = cleaned_outputs["X_val_full_xgb"]
X_test_full_xgb = cleaned_outputs["X_test_full_xgb"]

# Labels
y_train_f = np.asarray(cleaned_outputs["y_train_f"]).astype(int)
y_val_f = np.asarray(cleaned_outputs["y_val_f"]).astype(int)
y_test_f = np.asarray(cleaned_outputs["y_test_f"]).astype(int)

y_train_full = np.asarray(cleaned_outputs["y_train_full"]).astype(int)
y_val_full = np.asarray(cleaned_outputs["y_val_full"]).astype(int)
y_test_full = np.asarray(cleaned_outputs["y_test_full"]).astype(int)

print("Shapes:")
print("  Fund clean:", X_train_fund_clean.shape, X_val_fund_clean.shape, X_test_fund_clean.shape)
print("  Fund xgb:", X_train_fund_xgb.shape, X_val_fund_xgb.shape, X_test_fund_xgb.shape)
print("  Full xgb:", X_train_full_xgb.shape, X_val_full_xgb.shape, X_test_full_xgb.shape)
print("  Labels fund:", y_train_f.shape, y_val_f.shape, y_test_f.shape)
print("  Labels full:", y_train_full.shape, y_val_full.shape, y_test_full.shape)

assert X_train_fund_clean.shape[0] == y_train_f.shape[0]
assert X_val_fund_clean.shape[0] == y_val_f.shape[0]
assert X_test_fund_clean.shape[0] == y_test_f.shape[0]
assert X_train_full_xgb.shape[0] == y_train_full.shape[0]

Shapes:
  Fund clean: (886770, 51) (433890, 51) (442979, 51)
  Fund xgb: (886770, 51) (433890, 51) (442979, 51)
  Full xgb: (886770, 59) (433890, 59) (442979, 59)
  Labels fund: (886770,) (433890,) (442979,)
  Labels full: (886770,) (433890,) (442979,)


In [4]:
# Utilities (encoding + metrics + threshold)
def _build_ohe():
    """
    Fast OHE with rare-category grouping where available.
    """
    try:
        return OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=0.001,
            sparse_output=True,
            dtype=np.float32
        )
    except TypeError:
        # sklearn fallback
        try:
            return OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=0.001,
                sparse_output=True,
                dtype=np.float32
            )
        except TypeError:
            return OneHotEncoder(
                handle_unknown="ignore",
                sparse=True,
                dtype=np.float32
            )


def _sanitize_for_encoder(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for c in X.columns:
        col = X[c]
        if pd.api.types.is_numeric_dtype(col):
            X[c] = pd.to_numeric(col, errors="coerce")
        else:
            col_obj = col.astype("object")
            X[c] = col_obj.where(pd.notna(col_obj), np.nan)
    return X


def encode_categoricals(X_train, X_val, X_test):
    """
    Builds a sparse ColumnTransformer:
      - numeric: median impute (no scaling here)
      - categorical: constant impute + OHE
    """
    X_train = _sanitize_for_encoder(X_train)
    X_val = _sanitize_for_encoder(X_val)
    X_test = _sanitize_for_encoder(X_test)

    cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("ohe", _build_ohe())
    ])

    encoder = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=1.0  # force sparse output when possible
    )

    X_train_enc = encoder.fit_transform(X_train)
    X_val_enc = encoder.transform(X_val)
    X_test_enc = encoder.transform(X_test)

    return encoder, X_train_enc, X_val_enc, X_test_enc, num_cols, cat_cols


def transform_with_encoder(encoder, X):
    X = _sanitize_for_encoder(X)
    return encoder.transform(X)


def eval_scores(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
    }


def best_threshold_by_f1(y_true, p):
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    if len(thresholds) == 0:
        return 0.50, 0.0
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    idx = int(np.argmax(f1_vals))
    return float(thresholds[idx]), float(f1_vals[idx])

def metrics_at_threshold(y_true, p, thr):
    y_pred = (p >= thr).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

def fit_xgb_with_early_stopping(model, X_train, y_train, X_val, y_val, rounds=80):
    try:
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=rounds
        )
    except TypeError:
        # xgboost version compatibility fallback
        model.set_params(early_stopping_rounds=rounds)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    return model

In [5]:
# Encode the 3 model datasets
# Fundamental clean/xgb should have same columns
if list(X_train_fund_clean.columns) != list(X_train_fund_xgb.columns):
    raise ValueError("Fundamental clean and xgb columns differ. Cannot safely share encoder.")

# Fit fundamental encoder ONCE (on clean set), reuse for xgb set.
enc_fund_shared, X_train_fund_log_enc, X_val_fund_log_enc, X_test_fund_log_enc, num_f, cat_f = encode_categoricals(
    X_train_fund_clean, X_val_fund_clean, X_test_fund_clean
)

# Reuse same encoder for fund_xgb matrices (no second OHE fit)
X_train_fund_xgb_enc = transform_with_encoder(enc_fund_shared, X_train_fund_xgb)
X_val_fund_xgb_enc = transform_with_encoder(enc_fund_shared, X_val_fund_xgb)
X_test_fund_xgb_enc = transform_with_encoder(enc_fund_shared, X_test_fund_xgb)

# Keep key names consistent for downstream notebooks
enc_fund_log = enc_fund_shared
enc_fund_xgb = enc_fund_shared

# Full xgb encoder (separate, because feature set differs)
enc_full_xgb, X_train_full_xgb_enc, X_val_full_xgb_enc, X_test_full_xgb_enc, num_full_xgb, cat_full_xgb = encode_categoricals(
    X_train_full_xgb, X_val_full_xgb, X_test_full_xgb
)

print("Fund shared encoded shape:", X_train_fund_log_enc.shape)
print("Full xgb encoded shape:", X_train_full_xgb_enc.shape)

Fund shared encoded shape: (886770, 101)
Full xgb encoded shape: (886770, 141)


In [6]:
# Train the 3 models
# class imbalance helpers for XGB
neg_f = int((y_train_f == 0).sum())
pos_f = int((y_train_f == 1).sum())
scale_pos_weight_f = neg_f / max(pos_f, 1)
 
neg_full = int((y_train_full == 0).sum())
pos_full = int((y_train_full == 1).sum())
scale_pos_weight_full = neg_full / max(pos_full, 1)
 
print("scale_pos_weight_f:", round(scale_pos_weight_f, 4))
print("scale_pos_weight_full:", round(scale_pos_weight_full, 4))

# 1) Logistic Regression (Fundamental)
# Using MaxAbsScaler on sparse encoded matrix (fast + sparse-safe)
log_fund = Pipeline(steps=[
    ("maxabs", MaxAbsScaler()),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=5000,
        tol=1e-3,
        class_weight="balanced",
        random_state=42,
    ))
])
log_fund.fit(X_train_fund_log_enc, y_train_f)

# 2) XGB (Fundamental)
xgb_fund = XGBClassifier(
    n_estimators=1600,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight_f
)
xgb_fund = fit_xgb_with_early_stopping(
    xgb_fund, X_train_fund_xgb_enc, y_train_f, X_val_fund_xgb_enc, y_val_f, rounds=80
)

# 3) XGB (Full)
xgb_full = XGBClassifier(
    n_estimators=1600,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight_full
)
xgb_full = fit_xgb_with_early_stopping(
    xgb_full, X_train_full_xgb_enc, y_train_full, X_val_full_xgb_enc, y_val_full, rounds=80
)

print("Training complete.")

scale_pos_weight_f: 17.0172
scale_pos_weight_full: 17.0172


/Users/alex._choo/anaconda3/envs/cap5771/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training complete.


In [7]:
# Evaluate all models (val/test)
rows = []

# Logistic (Fundamental)
p_val_log = log_fund.predict_proba(X_val_fund_log_enc)[:, 1]
p_test_log = log_fund.predict_proba(X_test_fund_log_enc)[:, 1]
val_scores = eval_scores(y_val_f, p_val_log)
test_scores = eval_scores(y_test_f, p_test_log)
thr_log, val_best_f1_log = best_threshold_by_f1(y_val_f, p_val_log)
val_thr_metrics = metrics_at_threshold(y_val_f, p_val_log, thr_log)
test_thr_metrics = metrics_at_threshold(y_test_f, p_test_log, thr_log)
 
rows.append({
    "model": "Logistic (Fundamental)",
    **{f"val_{k}": v for k, v in val_scores.items()},
    **{f"test_{k}": v for k, v in test_scores.items()},
    "val_best_thr_f1": thr_log,
    "val_best_f1": val_best_f1_log,
    "val_accuracy_at_best_f1_thr": val_thr_metrics["accuracy"],
    "val_precision_at_best_f1_thr": val_thr_metrics["precision"],
    "val_recall_at_best_f1_thr": val_thr_metrics["recall"],
    "test_accuracy_at_val_best_f1_thr": test_thr_metrics["accuracy"],
    "test_precision_at_val_best_f1_thr": test_thr_metrics["precision"],
    "test_recall_at_val_best_f1_thr": test_thr_metrics["recall"],
    "test_f1_at_val_best_f1_thr": test_thr_metrics["f1"],
})


# XGB (Fundamental)
p_val_fund = xgb_fund.predict_proba(X_val_fund_xgb_enc)[:, 1]
p_test_fund = xgb_fund.predict_proba(X_test_fund_xgb_enc)[:, 1]
val_scores = eval_scores(y_val_f, p_val_fund)
test_scores = eval_scores(y_test_f, p_test_fund)
thr_fund, val_best_f1_fund = best_threshold_by_f1(y_val_f, p_val_fund)
val_thr_metrics = metrics_at_threshold(y_val_f, p_val_fund, thr_fund)
test_thr_metrics = metrics_at_threshold(y_test_f, p_test_fund, thr_fund)
 
rows.append({
    "model": "XGB (Fundamental)",
    **{f"val_{k}": v for k, v in val_scores.items()},
    **{f"test_{k}": v for k, v in test_scores.items()},
    "val_best_thr_f1": thr_fund,
    "val_best_f1": val_best_f1_fund,
    "val_accuracy_at_best_f1_thr": val_thr_metrics["accuracy"],
    "val_precision_at_best_f1_thr": val_thr_metrics["precision"],
    "val_recall_at_best_f1_thr": val_thr_metrics["recall"],
    "test_accuracy_at_val_best_f1_thr": test_thr_metrics["accuracy"],
    "test_precision_at_val_best_f1_thr": test_thr_metrics["precision"],
    "test_recall_at_val_best_f1_thr": test_thr_metrics["recall"],
    "test_f1_at_val_best_f1_thr": test_thr_metrics["f1"],
})


# XGB (Full)
p_val_full = xgb_full.predict_proba(X_val_full_xgb_enc)[:, 1]
p_test_full = xgb_full.predict_proba(X_test_full_xgb_enc)[:, 1]
val_scores = eval_scores(y_val_full, p_val_full)
test_scores = eval_scores(y_test_full, p_test_full)
thr_full, val_best_f1_full = best_threshold_by_f1(y_val_full, p_val_full)
val_thr_metrics = metrics_at_threshold(y_val_full, p_val_full, thr_full)
test_thr_metrics = metrics_at_threshold(y_test_full, p_test_full, thr_full)
 
rows.append({
    "model": "XGB (Full)",
    **{f"val_{k}": v for k, v in val_scores.items()},
    **{f"test_{k}": v for k, v in test_scores.items()},
    "val_best_thr_f1": thr_full,
    "val_best_f1": val_best_f1_full,
    "val_accuracy_at_best_f1_thr": val_thr_metrics["accuracy"],
    "val_precision_at_best_f1_thr": val_thr_metrics["precision"],
    "val_recall_at_best_f1_thr": val_thr_metrics["recall"],
    "test_accuracy_at_val_best_f1_thr": test_thr_metrics["accuracy"],
    "test_precision_at_val_best_f1_thr": test_thr_metrics["precision"],
    "test_recall_at_val_best_f1_thr": test_thr_metrics["recall"],
    "test_f1_at_val_best_f1_thr": test_thr_metrics["f1"],
})

summary = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
display(summary)

,model,val_roc_auc,val_pr_auc,val_brier,test_roc_auc,test_pr_auc,test_brier,val_best_thr_f1,val_best_f1,val_accuracy_at_best_f1_thr,val_precision_at_best_f1_thr,val_recall_at_best_f1_thr,test_accuracy_at_val_best_f1_thr,test_precision_at_val_best_f1_thr,test_recall_at_val_best_f1_thr,test_f1_at_val_best_f1_thr
0,XGB (Full),0.717182,0.155800,0.223519,0.712520,0.141501,0.220234,0.678941,0.230724,0.832342,0.167449,0.370865,0.842268,0.153577,0.346453,0.212816
1,XGB (Fundamental),0.672315,0.126593,0.219220,0.666881,0.110836,0.210933,0.598504,0.195307,0.790295,0.131990,0.375387,0.804704,0.118303,0.336806,0.175102
2,Logistic (Fundamental),0.625036,0.101383,0.081640,0.622717,0.090358,0.077727,0.202362,0.165183,0.656341,0.098875,0.501513,0.669054,0.089621,0.477991,0.150941


In [8]:
# Thresholded test reports using validation-chosen thresholds
model_threshold_plan = [
    ("Logistic (Fundamental)", y_test_f, p_test_log, thr_log),
    ("XGB (Fundamental)", y_test_f, p_test_fund, thr_fund),
    ("XGB (Full)", y_test_full, p_test_full, thr_full),
]
 
for name, y_true, p_test, thr in model_threshold_plan:
    y_pred = (p_test >= thr).astype(int)
    print("=" * 100)
    print(f"{name} | threshold={thr:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(
        "Accuracy:", round(accuracy_score(y_true, y_pred), 4),
        "Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1:", round(f1_score(y_true, y_pred, zero_division=0), 4)
    )
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4))

Logistic (Fundamental) | threshold=0.2024
Confusion matrix:
[[283346 132371]
 [ 14231  13031]]
Accuracy: 0.6691 Precision: 0.0896 Recall: 0.478 F1: 0.1509
Classification report:
              precision    recall  f1-score   support

           0     0.9522    0.6816    0.7945    415717
           1     0.0896    0.4780    0.1509     27262

    accuracy                         0.6691    442979
   macro avg     0.5209    0.5798    0.4727    442979
weighted avg     0.8991    0.6691    0.7549    442979

XGB (Fundamental) | threshold=0.5985
Confusion matrix:
[[347285  68432]
 [ 18080   9182]]
Accuracy: 0.8047 Precision: 0.1183 Recall: 0.3368 F1: 0.1751
Classification report:
              precision    recall  f1-score   support

           0     0.9505    0.8354    0.8892    415717
           1     0.1183    0.3368    0.1751     27262

    accuracy                         0.8047    442979
   macro avg     0.5344    0.5861    0.5322    442979
weighted avg     0.8993    0.8047    0.8453    44

In [10]:
# Check where notebook is running from
print("CWD:", Path.cwd())

# Robust path: if you're in notebooks/, go one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Saving to:", MODEL_DIR)

CWD: /Users/alex._choo/Desktop/CAP5771/CAP5771-Ychoo
Saving to: /Users/alex._choo/Desktop/CAP5771/CAP5771-Ychoo/artifacts/models


In [11]:
# Save trained models + encoders + summary in a bundle for easy loading in deployment or reference
trained_bundle = {
    "meta": meta,
    "summary": summary,

    # models
    "log_fund": log_fund,
    "xgb_fund": xgb_fund,
    "xgb_full": xgb_full,

    # encoders
    "enc_fund_log": enc_fund_log,
    "enc_fund_xgb": enc_fund_xgb,
    "enc_full_xgb": enc_full_xgb,

    # thresholds
    "thresholds": {
        "log_fund_val_best_f1_thr": float(thr_log),
        "xgb_fund_val_best_f1_thr": float(thr_fund),
        "xgb_full_val_best_f1_thr": float(thr_full),
    }
}

joblib.dump(trained_bundle, MODEL_DIR / "trained_models_bundle.joblib", compress=3)
summary.to_csv(MODEL_DIR / "model_summary.csv", index=False)

print("Saved:")
print(" -", MODEL_DIR / "trained_models_bundle.joblib")
print(" -", MODEL_DIR / "model_summary.csv")

Saved:
 - /Users/alex._choo/Desktop/CAP5771/CAP5771-Ychoo/artifacts/models/trained_models_bundle.joblib
 - /Users/alex._choo/Desktop/CAP5771/CAP5771-Ychoo/artifacts/models/model_summary.csv
